# Mini Project: Fashion Classifier
**A complete, realistic PyTorch project in one notebook** (companion to PyTorch Quest).

We train a small CNN to recognise 10 kinds of clothing (FashionMNIST, 70,000 tiny grayscale images). It is the "hello world" of deep learning, but built the way real repos are built: config, seeding, proper splits, augmentation, GPU, mixed precision, scheduler, gradient clipping, checkpoints, early stopping, resume, evaluation, error analysis and debugging checks.

**What you practice (and where it was in the Quest)**

| Section | Skill | Quest level |
|---|---|---|
| 1 Setup | device, seeds, config | 0, 8 |
| 2 Data | Dataset, transforms, splits, DataLoader | 1, 7 |
| 3 Model | nn.Module, CNN shapes, hooks | 5, 10, 11 |
| 4 Sanity checks | initial loss, overfit one batch | 12 |
| 5 Training | loop, AdamW, scheduler, AMP, clipping | 3, 6, 8 |
| 6 Checkpoints | save, load, resume | 9 |
| 7 Evaluation | no_grad, confusion matrix, error analysis | 6, 14 |
| 8 Debug lab | eval vs train mode, gradient norms | 12 |

**How to run:** Runtime > Change runtime type > GPU (T4), then Runtime > Run all. It takes a few minutes on GPU (much longer on CPU). Expected results are in comments; yours may differ slightly.

## 1. Setup: device, seeds, config

In [ ]:
import os, random, time, copy
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# One dict holds every knob. Real repos use argparse or YAML files for the same purpose.
CFG = dict(seed=42, batch_size=128, epochs=8, lr=1e-3, weight_decay=1e-4,
           val_size=5000, num_workers=2, clip_norm=1.0, patience=4, ckpt_dir="checkpoints")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG["seed"])

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"            # mixed precision only makes sense on a GPU
print(torch.__version__, "| device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
os.makedirs(CFG["ckpt_dir"], exist_ok=True)

> **Why seeds?** Random initial weights and shuffling make runs differ. Fixing the seed makes a run repeatable, which is how you tell "my change helped" from "I got lucky". (GPU runs can still differ slightly.)

## 2. Data: Dataset, transforms, split, DataLoader

In [ ]:
CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
MEAN, STD = 0.2860, 0.3530            # FashionMNIST pixel statistics (precomputed)

train_tf = transforms.Compose([       # augmentation: only for TRAINING data
    transforms.RandomCrop(28, padding=2),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),            # PIL image -> float tensor [1, 28, 28], values 0..1
    transforms.Normalize((MEAN,), (STD,)),
])
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,))])

root = "data"
full_aug   = datasets.FashionMNIST(root, train=True,  download=True, transform=train_tf)
full_plain = datasets.FashionMNIST(root, train=True,  download=True, transform=eval_tf)
test_ds    = datasets.FashionMNIST(root, train=False, download=True, transform=eval_tf)

# Train/validation split by index. Two dataset objects, same files, different transforms:
# this keeps augmentation OUT of validation (random_split would share one transform).
g = torch.Generator().manual_seed(CFG["seed"])
perm = torch.randperm(len(full_aug), generator=g).tolist()
train_ds = Subset(full_aug,   perm[:-CFG["val_size"]])
val_ds   = Subset(full_plain, perm[-CFG["val_size"]:])
print(len(train_ds), len(val_ds), len(test_ds))          # 55000 5000 10000

pin = device == "cuda"
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=pin, generator=g)
val_loader   = DataLoader(val_ds,  batch_size=256, shuffle=False, num_workers=CFG["num_workers"], pin_memory=pin)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=CFG["num_workers"], pin_memory=pin)

xb, yb = next(iter(train_loader))
print(xb.shape, xb.dtype)             # torch.Size([128, 1, 28, 28]) torch.float32   [B, C, H, W]
print(yb.shape, yb.dtype)             # torch.Size([128]) torch.int64                labels are long
print(len(train_loader))              # 430 batches (55000 / 128, rounded up)

In [ ]:
def show_images(x, titles, cols=8):
    rows = (len(x) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.9))
    for ax in np.array(axes).flat: ax.axis("off")
    for ax, img, t in zip(np.array(axes).flat, x, titles):
        ax.imshow((img.squeeze(0) * STD + MEAN).clamp(0, 1), cmap="gray")   # undo Normalize to look at it
        ax.set_title(t, fontsize=8)
    plt.tight_layout(); plt.show()

show_images(xb[:16], [CLASSES[i] for i in yb[:16].tolist()])

> **Data pipeline in one line:** raw files > `FashionMNIST` Dataset (+ transforms) > `Subset` (split) > `DataLoader` (batches, shuffling, workers) > `[B,1,28,28]` tensor > model. Whenever you read a research repo, find these five steps first.

## 3. Model: a small CNN, and tracing its shapes

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10, p_drop=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),   # [B,32,14,14]
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),  # [B,64,7,7]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                   # [B, 64*7*7 = 3136]
            nn.Dropout(p_drop),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),          # [B, 128]
            nn.Linear(128, n_classes),                      # [B, 10]  raw scores (logits), no softmax here
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def trace_shapes(model, input_shape):
    """Print input/output shape of every leaf layer using forward hooks (Quest level 11)."""
    rows, handles = [], []
    for name, m in model.named_modules():
        if len(list(m.children())) == 0:
            def hook(mod, inp, out, name=name):
                rows.append((name, mod.__class__.__name__, tuple(inp[0].shape), tuple(out.shape)))
            handles.append(m.register_forward_hook(hook))
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(*input_shape, device=next(model.parameters()).device))
    for h in handles: h.remove()
    model.train(was_training)
    for n, c, i, o in rows:
        print(f"{n:<14}{c:<12}{str(i):<20}-> {o}")

model = SmallCNN().to(device)
print("trainable parameters:", count_params(model))        # 421,834
trace_shapes(model, (1, 1, 28, 28))

> **Check the parameter count by hand:** conv1 `1*32*9+32 = 320`, BN `2*32 = 64`, conv2 `32*64*9+64 = 18,496`, BN `2*64 = 128`, fc1 `3136*128+128 = 401,536`, fc2 `128*10+10 = 1,290`. Sum: **421,834**. Almost all weights sit in the first Linear layer, a typical pattern after flattening.

## 4. Sanity checks (do these BEFORE training for real)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

# Check 1: an untrained 10-class model should have loss near ln(10) = 2.303 (guessing uniformly).
model.eval()
with torch.no_grad():
    print("initial loss:", loss_fn(model(xb.to(device)), yb.to(device)).item(), "| ln(10) =", np.log(10))
    # expect something close to 2.3. Far off? Check labels, output size, or a missing/extra softmax.

# Check 2: can the model memorise ONE small batch? If not, the bug is in the code, not the data.
tiny = SmallCNN().to(device)
tiny_opt = torch.optim.Adam(tiny.parameters(), lr=1e-3)
x_small, y_small = xb[:64].to(device), yb[:64].to(device)
tiny.train()
for step in range(60):
    loss = loss_fn(tiny(x_small), y_small)
    tiny_opt.zero_grad(); loss.backward(); tiny_opt.step()
    if step % 20 == 0 or step == 59:
        print(step, round(loss.item(), 4))             # falls from about 2.3 to near 0

> If check 2 does not collapse toward zero, stop and debug (wrong labels, forgotten `zero_grad`, learning rate, detached tensor) before spending GPU time on full training.

## 5. Training: loop, AdamW, scheduler, mixed precision, clipping

In [ ]:
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)        # scales the loss so fp16 gradients do not underflow

def run_epoch(model, loader, loss_fn, opt=None, scaler=None, clip=None):
    """One pass over a loader. Training if an optimizer is given, otherwise evaluation."""
    training = opt is not None
    model.train(training)                                      # train(): dropout on, BN uses batch stats
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):                     # no graph when evaluating
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logits = model(xb)
                loss = loss_fn(logits, yb)
            if training:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                if clip:
                    scaler.unscale_(opt)                       # un-scale gradients before clipping them
                    nn.utils.clip_grad_norm_(model.parameters(), clip)
                scaler.step(opt)
                scaler.update()
            total_loss += loss.item() * xb.size(0)             # weight by batch size (last batch is smaller)
            correct += (logits.argmax(1) == yb).sum().item()
            n += xb.size(0)
    return total_loss / n, correct / n

def save_ckpt(path, model, opt, sched, scaler, next_epoch, best_acc, history):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": next_epoch, "best_acc": best_acc,
                "history": history, "cfg": CFG}, path)

def fit(model, opt, sched, scaler, history, epochs, start_epoch=0, best_acc=0.0):
    bad = 0
    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, loss_fn, opt, scaler, CFG["clip_norm"])
        va_loss, va_acc = run_epoch(model, val_loader, loss_fn)
        history["lr"].append(opt.param_groups[0]["lr"])
        sched.step()                                           # update the learning rate once per epoch
        for k, v in zip(("train_loss", "train_acc", "val_loss", "val_acc"), (tr_loss, tr_acc, va_loss, va_acc)):
            history[k].append(v)
        if va_acc > best_acc:
            best_acc, bad = va_acc, 0
            torch.save(model.state_dict(), f"{CFG['ckpt_dir']}/best.pt")        # weights only: for inference
        else:
            bad += 1
        save_ckpt(f"{CFG['ckpt_dir']}/last.pt", model, opt, sched, scaler, epoch + 1, best_acc, history)  # everything: to resume
        print(f"epoch {epoch+1:>2}/{epochs}  train {tr_loss:.3f}/{tr_acc:.3f}  val {va_loss:.3f}/{va_acc:.3f}  "
              f"lr {history['lr'][-1]:.5f}  {time.time()-t0:.0f}s")
        if bad >= CFG["patience"]:
            print("early stopping: no validation improvement"); break
    return best_acc

seed_everything(CFG["seed"])
model = SmallCNN().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
best_acc = fit(model, opt, sched, scaler, history, CFG["epochs"])
print("best val acc:", best_acc)                            # roughly 0.91 to 0.93

> **Read the training loop like a map.** Each epoch: `run_epoch(train)` = forward, loss, `zero_grad`, `backward`, `step` (Quest level 6) > `run_epoch(val)` under `no_grad` > scheduler step > save checkpoints. Everything else (AMP, clipping, early stopping) is a safety or speed layer around that heartbeat.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val"); ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="val"); ax[1].set_title("accuracy"); ax[1].legend()
ax[2].plot(history["lr"]); ax[2].set_title("learning rate (cosine)")
plt.tight_layout(); plt.show()
# Healthy: both losses fall. Train accuracy above val accuracy by a few points = mild overfitting (normal).

## 6. Checkpoints: verify them, then resume

In [ ]:
# Verify: a model rebuilt from last.pt must give identical outputs to the one in memory.
ck = torch.load(f"{CFG['ckpt_dir']}/last.pt", map_location=device, weights_only=False)   # our own file, so we trust it
print(list(ck.keys()))            # ['model', 'opt', 'sched', 'scaler', 'epoch', 'best_acc', 'history', 'cfg']
print("next epoch:", ck["epoch"], "| best acc:", ck["best_acc"])

m2 = SmallCNN().to(device)
m2.load_state_dict(ck["model"])
model.eval(); m2.eval()
with torch.no_grad():
    x = xb[:16].to(device)
    print("identical outputs:", torch.allclose(model(x), m2(x), atol=1e-5))     # True

In [ ]:
# Resume: rebuild EVERYTHING (model, optimizer, scheduler, scaler, history) and continue 1 extra epoch.
model_r = SmallCNN().to(device)
opt_r = torch.optim.AdamW(model_r.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched_r = torch.optim.lr_scheduler.CosineAnnealingLR(opt_r, T_max=CFG["epochs"])
scaler_r = torch.amp.GradScaler("cuda", enabled=use_amp)

model_r.load_state_dict(ck["model"]); opt_r.load_state_dict(ck["opt"])
sched_r.load_state_dict(ck["sched"]); scaler_r.load_state_dict(ck["scaler"])
hist_r = ck["history"]
fit(model_r, opt_r, sched_r, scaler_r, hist_r, epochs=ck["epoch"] + 1, start_epoch=ck["epoch"], best_acc=ck["best_acc"])
# One more line appears. (The cosine schedule is past its planned end, so the lr bounces up slightly; fine for a demo.)

> **Two kinds of checkpoint.** `best.pt` = weights only (what you ship or load for inference, like the pretrained model in the DQN project). `last.pt` = weights + optimizer + scheduler + epoch + history (what you need to *resume training* without a hiccup).

## 7. Evaluation: test set, confusion matrix, error analysis

In [ ]:
model = SmallCNN().to(device)
model.load_state_dict(torch.load(f"{CFG['ckpt_dir']}/best.pt", map_location=device))
model.eval()

test_loss, test_acc = run_epoch(model, test_loader, loss_fn)
print(f"test loss {test_loss:.3f} | test acc {test_acc:.3f}")    # close to the val accuracy (about 0.91-0.93)

@torch.no_grad()
def collect(model, loader):
    model.eval()
    P, Y, C = [], [], []
    for xb_, yb_ in loader:
        probs = F.softmax(model(xb_.to(device)), dim=1).cpu()    # softmax turns logits into probabilities
        conf, pred = probs.max(1)
        P.append(pred); Y.append(yb_); C.append(conf)
    return torch.cat(P), torch.cat(Y), torch.cat(C)

preds, labels, confs = collect(model, test_loader)
cm = torch.bincount(labels * 10 + preds, minlength=100).reshape(10, 10)   # rows = true class, cols = predicted
per_class = cm.diag().float() / cm.sum(1)
for i in per_class.argsort():
    print(f"{CLASSES[i]:<14}{per_class[i]:.3f}")
# Typically worst: Shirt, T-shirt/top, Pullover, Coat (they look alike). Best: Trouser, Bag, Sneaker, Ankle boot.

In [ ]:
plt.figure(figsize=(6.5, 5.5))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(10), CLASSES, rotation=90); plt.yticks(range(10), CLASSES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar(); plt.tight_layout(); plt.show()

# The model's most confident MISTAKES: often label noise or genuinely ambiguous items.
wrong = (preds != labels).nonzero().squeeze(1)
worst = wrong[confs[wrong].argsort(descending=True)][:8].tolist()
show_images([test_ds[i][0] for i in worst],
            [f"{CLASSES[labels[i]]} -> {CLASSES[preds[i]]} ({confs[i]:.2f})" for i in worst], cols=4)

In [ ]:
# Inference on ONE image: add the batch dimension, no_grad, softmax, top-3.
x, y = test_ds[0]                                        # x: [1, 28, 28] (no batch dim yet)
with torch.no_grad():
    p = F.softmax(model(x.unsqueeze(0).to(device)), dim=1)[0]      # unsqueeze(0): [1,1,28,28]
top = p.topk(3)
print("true label:", CLASSES[y])
for prob, idx in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{CLASSES[idx]:<14}{prob:.3f}")

## 8. Debug lab: break things on purpose

In [ ]:
# Experiment A: forgetting model.eval(). Same weights, different mode.
def acc_in_mode(m, loader, train_mode):
    m.train(train_mode)
    correct = n = 0
    with torch.no_grad():
        for xb_, yb_ in loader:
            xb_, yb_ = xb_.to(device), yb_.to(device)
            correct += (m(xb_).argmax(1) == yb_).sum().item(); n += len(yb_)
    return correct / n

print("eval mode :", acc_in_mode(copy.deepcopy(model), val_loader, False))   # the honest number
print("train mode:", acc_in_mode(copy.deepcopy(model), val_loader, True))    # usually lower and noisier
# Dropout randomly zeroes activations and BatchNorm uses per-batch statistics in train mode.
# (We used deepcopy because train mode under no_grad still updates BatchNorm's running statistics!)

In [ ]:
# Experiment B: gradient norms per parameter. A healthy network has gradients of similar magnitude.
dbg = SmallCNN().to(device); dbg.train()
loss = loss_fn(dbg(xb.to(device)), yb.to(device))
loss.backward()
for name, p in dbg.named_parameters():
    print(f"{name:<24}{tuple(p.shape)!s:<18}grad norm {p.grad.norm().item():.4f}")
# Many zeros or huge values (1e3+) hint at vanishing or exploding gradients.

In [ ]:
# Experiment C: the SAME images, with and without augmentation, give different numbers.
idx = perm[:5000]                                  # 5000 training images (so both numbers are optimistic)
for name, ds in [("plain", Subset(full_plain, idx)), ("augmented", Subset(full_aug, idx))]:
    dl = DataLoader(ds, batch_size=256, shuffle=False, num_workers=CFG["num_workers"])
    print(f"{name:<10}", round(run_epoch(model, dl, loss_fn)[1], 4))
# Usually "augmented" is a bit lower (random crops and flips make the task harder).
# Lesson: transforms are part of the measurement. Always evaluate on un-augmented data.

## 9. How this maps to your DQN TurtleBot project

| This notebook | In the DQN project |
|---|---|
| `SmallCNN` (an `nn.Module`) | the Q-network: state in, one score per action out |
| `Dataset` + `DataLoader` | the replay buffer: random batches of past experience |
| `CrossEntropyLoss` vs labels | Huber loss between Q(s,a) and the Bellman target |
| `run_epoch` training branch | the DQN update step (zero_grad, backward, step) |
| `best.pt` + `load_state_dict` | loading the pretrained model for `dqn_test` |
| `model.eval()` + `no_grad` + `argmax` | acting greedily at test time |
| `device` + `map_location="cpu"` | running the model in your VM |
| `CFG` dict | hyperparameters (gamma, epsilon, lr, batch size) in the repo |

## 10. Challenges (do them in order, one per sitting)
1. **Swap the architecture.** Replace the CNN with an MLP (`Flatten`, `Linear(784,256)`, `ReLU`, `Linear(256,10)`). Compare parameter count, accuracy and training time.
2. **Ablate.** Remove BatchNorm, then dropout, then augmentation, one at a time. Which hurts most? Use the same seed.
3. **Optimizer lab.** Try `SGD(lr=0.05, momentum=0.9)` against AdamW. Plot validation accuracy for both.
4. **Learning-rate scan.** Train 2 epochs each with lr = 1e-1, 1e-2, 1e-3, 1e-4. Which diverges? Which crawls?
5. **Add a residual block** (Quest level 11) between the two conv stages. Keep the channel count equal so `x + f(x)` works.
6. **Bigger problem.** Switch to `datasets.CIFAR10` (3 channels, 32x32). Only a few numbers change: input channels, the Mean/Std, and the flatten size (hint: print the shape after `features` using `trace_shapes`).
7. **Break and diagnose.** Delete `opt.zero_grad` from `run_epoch`, or move the data to the wrong device, and use the Section 4 and 8 tools to find it before reading the error message.

When challenge 6 feels easy, you are ready to read research repos. When you can explain every cell above without scrolling back, you are ready for the DQN code.